# Standard Out/Error Logging in MPI Codes

**TODO:** Write text that explains this example.

In [1]:
# Remove input/outputs in rendering

import subprocess as sbp

from pathlib import Path
from IPython.display import Code

import poptus

N_MPI_PROCESSES = 4
MPI_EXEC = Path.cwd().joinpath("example_mpi_program.py")

In [2]:
# Remove input in rendering

Code(filename=MPI_EXEC, language='python')

#!/usr/bin/env python

import sys

from mpi4py import MPI

import poptus

LEAD_PROCESS = 0

# Users specify the log level
verbosity = int(sys.argv[1])

# Standard MPI setup
mpi_comm = MPI.COMM_WORLD
comm_size = mpi_comm.Get_size()
rank = mpi_comm.Get_rank()
is_lead = (rank == LEAD_PROCESS)

# Create MPI-aware log functions
logger = poptus.create_logger({"Level": verbosity}, rank=rank, is_lead=is_lead)
log, log_debug, warn, log_and_abort = \
    poptus.create_log_functions(logger, "MpiTest")

# Log general example information
log(f"Running MPI example with {comm_size} processes")
log(f"The logging process is rank {rank}")
log(f"Logging at verbosity level {verbosity}")

# Simple MPI execution
if rank == LEAD_PROCESS:
    workers = [i for i in range(comm_size) if i != LEAD_PROCESS]
    tasks_all = [1.1 * i for i in range(len(workers))]

    for i, task in zip(workers, tasks_all):
        log_debug(f"Sending task to the rank {i} worker process",
                  poptus.LOG_LEVEL_MIN_DEBUG)
        mpi_comm.send(task, dest=i, tag=1)
else:
    task = mpi_comm.recv(source=LEAD_PROCESS, tag=1)
    log_debug(f"Received task {task} from lead process",
              poptus.LOG_LEVEL_MIN_DEBUG)

    if task == 1.1:
        warn("Are you sure that's the task I should work on?")

In [5]:
# Remove input in rendering

CMD = ["mpirun", "-np", str(N_MPI_PROCESSES), MPI_EXEC, str(poptus.LOG_LEVEL_DEFAULT)]
result = sbp.run(CMD, check=True)
assert result.returncode == 0
# TODO: Sanity check of arguments so that this notebook has some level of testing of
# the correctness of its contents?

[MpiTest] Running MPI example with 4 processes
[MpiTest] The logging process is rank 0
[MpiTest] Logging at verbosity level 1
[Rank 2] WARNING - Are you sure that's the task I should work on?


In [4]:
# Remove input in rendering

CMD = ["mpirun", "-np", str(N_MPI_PROCESSES), MPI_EXEC, str(poptus.LOG_LEVEL_MIN_DEBUG)]
result = sbp.run(CMD, check=True)
assert result.returncode == 0

[MpiTest] Running MPI example with 4 processes
[MpiTest] The logging process is rank 0
[MpiTest] Logging at verbosity level 2
[MpiTest] Sending task to the rank 1 worker process
[MpiTest] Sending task to the rank 2 worker process
[MpiTest] Sending task to the rank 3 worker process
[Rank 1] Received task 0.0 from lead process
[Rank 2] Received task 1.1 from lead process
[Rank 2] WARNING - Are you sure that's the task I should work on?
[Rank 3] Received task 2.2 from lead process
